In [1]:
import pandas as pd
import numpy as np
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTE

In [2]:
data = pd.read_csv(
    r"C:\\Users\\Haarun\\obesity_project\\data\\Obesity_Dataset.tsv",
    sep="\t"
)

In [3]:
print(data.head())

print(data.columns)

print(data.shape)

  DATE ADDED TO CATALOG  PUBMEDID FIRST AUTHOR        DATE            JOURNAL  \
0            2018-11-05  30120429  Schlauch KA  2018-08-17  Int J Obes (Lond)   
1            2018-11-05  30120429  Schlauch KA  2018-08-17  Int J Obes (Lond)   
2            2018-11-05  30120429  Schlauch KA  2018-08-17  Int J Obes (Lond)   
3            2018-11-05  30120429  Schlauch KA  2018-08-17  Int J Obes (Lond)   
4            2018-11-05  30120429  Schlauch KA  2018-08-17  Int J Obes (Lond)   

                                   LINK  \
0  www.ncbi.nlm.nih.gov/pubmed/30120429   
1  www.ncbi.nlm.nih.gov/pubmed/30120429   
2  www.ncbi.nlm.nih.gov/pubmed/30120429   
3  www.ncbi.nlm.nih.gov/pubmed/30120429   
4  www.ncbi.nlm.nih.gov/pubmed/30120429   

                                               STUDY  \
0  Single-nucleotide polymorphisms in a cohort of...   
1  Single-nucleotide polymorphisms in a cohort of...   
2  Single-nucleotide polymorphisms in a cohort of...   
3  Single-nucleotide polymorph

In [4]:
data["Target"] = data["P-VALUE"].apply(
    lambda x: 1 if x < 1e-6 else 0
)
print(data["Target"].value_counts())

Target
1    1426
0     115
Name: count, dtype: int64


In [5]:
features = [
    "CHR_ID",
    "CHR_POS",
    "REPORTED GENE(S)",
    "MAPPED_GENE",
    "RISK ALLELE FREQUENCY",
    "OR or BETA",
    "STRONGEST SNP-RISK ALLELE",
    "SNPS",
    "CONTEXT",
    "INTERGENIC",
    "UPSTREAM_GENE_DISTANCE",
    "DOWNSTREAM_GENE_DISTANCE"
]

X = data[features]

y = data["Target"]

In [6]:
X = X.fillna("Unknown")

In [7]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

categorical_columns = [
    "REPORTED GENE(S)",
    "MAPPED_GENE",
    "STRONGEST SNP-RISK ALLELE",
    "SNPS",
    "CONTEXT",
    "INTERGENIC"
]

for col in categorical_columns:
    X[col] = label_encoder.fit_transform(
        X[col].astype(str)
    )

In [8]:
numeric_columns = [
    "CHR_ID",
    "CHR_POS",
    "RISK ALLELE FREQUENCY",
    "OR or BETA",
    "UPSTREAM_GENE_DISTANCE",
    "DOWNSTREAM_GENE_DISTANCE"
]

for col in numeric_columns:
    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )

X = X.fillna(0)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [12]:
print(y_train_smote.value_counts())

Target
1    1139
0    1139
Name: count, dtype: int64


In [13]:
from sklearn.decomposition import PCA

In [14]:
pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X_scaled)

In [15]:
print("Original shape:", X_scaled.shape)

print("Reduced shape:", X_pca.shape)

Original shape: (1541, 12)
Reduced shape: (1541, 10)


In [16]:
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
smote = SMOTE(random_state=42)

X_train_pca_smote, y_train_pca_smote = smote.fit_resample(
    X_train_pca,
    y_train_pca
)

In [19]:
from xgboost import XGBClassifier

In [23]:
xgb_pca = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

xgb_pca.fit(
    X_train_pca_smote,
    y_train_pca_smote
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [24]:
y_pred_pca = xgb_pca.predict(X_test_pca)

y_prob_pca = xgb_pca.predict_proba(X_test_pca)[:, 1]

print("===== PCA + XGBOOST RESULTS =====")

print("Accuracy:", accuracy_score(y_test_pca, y_pred_pca))

print(classification_report(y_test_pca, y_pred_pca))

print("ROC-AUC:", roc_auc_score(y_test_pca, y_prob_pca))

===== PCA + XGBOOST RESULTS =====
Accuracy: 0.8932038834951457
              precision    recall  f1-score   support

           0       0.36      0.64      0.46        22
           1       0.97      0.91      0.94       287

    accuracy                           0.89       309
   macro avg       0.66      0.77      0.70       309
weighted avg       0.93      0.89      0.91       309

ROC-AUC: 0.8905606588533418
